# Reproducible Builds and Environment Checks
Companion notebook for the **Reproducible Builds and Environment Checks** section of Chapter 7.

Pinned dependencies aren't enough on their own -- the *same setup* has to run every time. This notebook runs the book's own build-and-smoke-check commands for real against the scaffold, then shows the matching CI workflow.

As in notebook 03, every command below uses `sys.executable` explicitly instead of a bare `python`/`pip`, so it always runs against this notebook's own kernel environment.

In [1]:
import sys

!"{sys.executable}" --version

Python 3.13.5


In [2]:
!cd agent-app && "{sys.executable}" -m pip install --quiet --upgrade pip

In [3]:
!cd agent-app && "{sys.executable}" -m pip install --quiet -r requirements.txt

In [4]:
!cd agent-app && "{sys.executable}" -m pip install --quiet -e .

This last command is only a **smoke check** -- it doesn't prove full reproducibility, but it catches environment drift early by showing the package installs and starts:

In [5]:
!cd agent-app && "{sys.executable}" -m support_agent --help

usage: support_agent [-h] [--message MESSAGE] [--customer-id CUSTOMER_ID]

options:
  -h, --help            show this help message and exit
  --message MESSAGE
  --customer-id CUSTOMER_ID


Run the *same* checks in CI. `agent-app/.github/workflows/environment-check.yml` is the book's GitHub Actions example, completed into a valid workflow (checkout -> set up Python 3.11 -> install -> smoke check -> tests):

In [6]:
print(open("agent-app/.github/workflows/environment-check.yml").read())

name: Environment Check

on:
  push:
  pull_request:

jobs:
  build:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt
          pip install -e .

      - name: Smoke check
        run: python -m support_agent --help

      - name: Run tests
        run: pip install -e ".[test]" && pytest



A small team should be able to answer four questions from files like these alone: which Python version (`3.11`, pinned in the workflow), which lock file (`requirements.txt`), what installs the app (`pip install -e .`), and what starts it (`python -m support_agent`).